# Практична робота №3
## SQL-аналітика очищених IoT-даних у DuckDB

Input: `results/practical_02/clean_iot_events.parquet`.
Output: `results/practical_03/`.


## Дані студента

**ПІБ:** _вкажіть тут_  
**Група:** _вкажіть тут_  
**Дата:** _вкажіть тут_


In [ ]:
from pathlib import Path
import json
import duckdb
import polars as pl

ROOT = Path('/workspace') if Path('/workspace').exists() else Path.cwd()
if not (ROOT / 'data' / 'input').exists() and (ROOT.parent / 'data' / 'input').exists():
    ROOT = ROOT.parent

INPUT_DIR = ROOT / 'data' / 'input'
PRACTICAL_02_RESULTS_DIR = ROOT / 'results' / 'practical_02'
PRACTICAL_03_RESULTS_DIR = ROOT / 'results' / 'practical_03'
PRACTICAL_03_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_EVENTS_PATH = PRACTICAL_02_RESULTS_DIR / 'clean_iot_events.parquet'
METADATA_PATH = INPUT_DIR / 'metadata.json'
DEVICE_REGISTRY_PATH = INPUT_DIR / 'device_registry.csv'
METRIC_CATALOG_PATH = INPUT_DIR / 'metric_catalog.csv'
DEVICE_TYPE_METRICS_PATH = INPUT_DIR / 'device_type_metrics.csv'

DEVICE_ACTIVITY_PATH = PRACTICAL_03_RESULTS_DIR / 'device_activity_summary.csv'
HOURLY_METRIC_PATH = PRACTICAL_03_RESULTS_DIR / 'hourly_metric_summary.csv'
INDICATORS_PATH = PRACTICAL_03_RESULTS_DIR / 'top_activity_indicators.csv'
SUMMARY_PATH = PRACTICAL_03_RESULTS_DIR / 'practical_03_summary.json'
QUERIES_PATH = PRACTICAL_03_RESULTS_DIR / 'practical_03_queries.sql'


In [ ]:
required_files = [CLEAN_EVENTS_PATH, METADATA_PATH, DEVICE_REGISTRY_PATH, METRIC_CATALOG_PATH, DEVICE_TYPE_METRICS_PATH]
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(missing_files)
with METADATA_PATH.open('r', encoding='utf-8') as f:
    metadata = json.load(f)
metadata


In [ ]:
con = duckdb.connect(database=':memory:')
con.execute(f"CREATE OR REPLACE VIEW clean_events AS SELECT * FROM read_parquet('{CLEAN_EVENTS_PATH.as_posix()}')")
con.execute(f"CREATE OR REPLACE VIEW devices AS SELECT * FROM read_csv_auto('{DEVICE_REGISTRY_PATH.as_posix()}', header=true)")
con.execute(f"CREATE OR REPLACE VIEW metric_catalog AS SELECT * FROM read_csv_auto('{METRIC_CATALOG_PATH.as_posix()}', header=true)")
con.execute(f"CREATE OR REPLACE VIEW device_type_metrics AS SELECT * FROM read_csv_auto('{DEVICE_TYPE_METRICS_PATH.as_posix()}', header=true)")
def sql_to_polars(sql: str) -> pl.DataFrame:
    return con.sql(sql).pl()
sql_to_polars('SELECT * FROM clean_events LIMIT 5')


## 1. Базовий SQL-профіль
Дописати `BASE_PROFILE_SQL` та `EVENT_TYPE_COUNTS_SQL`.


In [ ]:
BASE_PROFILE_SQL = '''
SELECT
    COUNT(*) AS clean_records
    -- TODO: devices_in_events
    -- TODO: metrics_in_events
    -- TODO: event_types
    -- TODO: first_event_ts
    -- TODO: last_event_ts
FROM clean_events
'''
base_profile_df = sql_to_polars(BASE_PROFILE_SQL)
base_profile_df


In [ ]:
EVENT_TYPE_COUNTS_SQL = '''
SELECT
    -- TODO: event_type
    -- TODO: events_count
FROM clean_events
-- TODO: GROUP BY
-- TODO: ORDER BY
'''
event_type_counts_df = sql_to_polars(EVENT_TYPE_COUNTS_SQL)
event_type_counts_df


## 2. Профіль активності пристроїв


In [ ]:
DEVICE_ACTIVITY_SQL = '''
SELECT
    e.device_id,
    d.device_type,
    d.location,
    COUNT(*) AS events_count
    -- TODO: first_event_ts
    -- TODO: last_event_ts
    -- TODO: unique_metrics
    -- TODO: active_hours
    -- TODO: avg_events_per_hour
FROM clean_events e
JOIN devices d ON e.device_id = d.device_id
GROUP BY e.device_id, d.device_type, d.location
ORDER BY events_count DESC, e.device_id ASC
'''
device_activity_df = sql_to_polars(DEVICE_ACTIVITY_SQL)
device_activity_df


In [ ]:
DEVICE_ACTIVITY_COLUMNS = ['device_id','device_type','location','events_count','first_event_ts','last_event_ts','unique_metrics','active_hours','avg_events_per_hour']
missing = [col for col in DEVICE_ACTIVITY_COLUMNS if col not in device_activity_df.columns]
if missing:
    raise ValueError(missing)
device_activity_df.select(DEVICE_ACTIVITY_COLUMNS).write_csv(DEVICE_ACTIVITY_PATH)
DEVICE_ACTIVITY_PATH


## 3. Погодинна агрегація метрик


In [ ]:
HOURLY_METRIC_SQL = '''
SELECT
    date_trunc('hour', e.event_ts) AS hour,
    e.device_id
    -- TODO: device_type
    -- TODO: location
    -- TODO: metric
    -- TODO: events_count
    -- TODO: avg_value
    -- TODO: min_value
    -- TODO: max_value
FROM clean_events e
JOIN devices d ON e.device_id = d.device_id
-- TODO: GROUP BY
-- TODO: ORDER BY
'''
hourly_metric_df = sql_to_polars(HOURLY_METRIC_SQL)
hourly_metric_df


In [ ]:
HOURLY_METRIC_COLUMNS = ['hour','device_id','device_type','location','metric','events_count','avg_value','min_value','max_value']
missing = [col for col in HOURLY_METRIC_COLUMNS if col not in hourly_metric_df.columns]
if missing:
    raise ValueError(missing)
hourly_metric_df.select(HOURLY_METRIC_COLUMNS).write_csv(HOURLY_METRIC_PATH)
HOURLY_METRIC_PATH


## 4. Аналітичні індикатори


In [ ]:
ACTIVITY_INDICATORS_SQL = '''
WITH device_activity AS (
    SELECT e.device_id, d.device_type, d.location, COUNT(*) AS events_count
    FROM clean_events e
    JOIN devices d ON e.device_id = d.device_id
    GROUP BY e.device_id, d.device_type, d.location
),
top_devices AS (
    SELECT 'top_device_by_events' AS indicator_type,
           ROW_NUMBER() OVER (ORDER BY events_count DESC, device_id ASC) AS rank,
           device_id, device_type, location,
           CAST(NULL AS TIMESTAMP) AS hour,
           CAST(NULL AS VARCHAR) AS metric,
           CAST(events_count AS DOUBLE) AS indicator_value,
           CAST(NULL AS DOUBLE) AS baseline_value,
           'top event count' AS details
    FROM device_activity
    QUALIFY rank <= 5
)
SELECT * FROM top_devices
-- TODO: додайте інші indicator_type через UNION ALL
ORDER BY indicator_type, rank
'''
activity_indicators_df = sql_to_polars(ACTIVITY_INDICATORS_SQL)
activity_indicators_df


In [ ]:
INDICATOR_COLUMNS = ['indicator_type','rank','device_id','device_type','location','hour','metric','indicator_value','baseline_value','details']
missing = [col for col in INDICATOR_COLUMNS if col not in activity_indicators_df.columns]
if missing:
    raise ValueError(missing)
activity_indicators_df.select(INDICATOR_COLUMNS).write_csv(INDICATORS_PATH)
INDICATORS_PATH


## 5. Збереження SQL-запитів і summary


In [ ]:
queries = {'BASE_PROFILE_SQL': BASE_PROFILE_SQL, 'EVENT_TYPE_COUNTS_SQL': EVENT_TYPE_COUNTS_SQL, 'DEVICE_ACTIVITY_SQL': DEVICE_ACTIVITY_SQL, 'HOURLY_METRIC_SQL': HOURLY_METRIC_SQL, 'ACTIVITY_INDICATORS_SQL': ACTIVITY_INDICATORS_SQL}
with QUERIES_PATH.open('w', encoding='utf-8') as f:
    for name, query in queries.items():
        f.write(f'-- {name}\n{query.strip()}\n\n')
QUERIES_PATH


In [ ]:
event_type_counts = {row[0]: row[1] for row in con.execute('SELECT event_type, COUNT(*) FROM clean_events GROUP BY event_type').fetchall()}
summary = {
    'variant_id': metadata.get('variant_id'),
    'student_id': metadata.get('student_id'),
    'student_name': metadata.get('student_name'),
    'clean_records': int(con.execute('SELECT COUNT(*) FROM clean_events').fetchone()[0]),
    'devices_in_events': int(con.execute('SELECT COUNT(DISTINCT device_id) FROM clean_events').fetchone()[0]),
    'metrics_in_events': int(con.execute('SELECT COUNT(DISTINCT metric) FROM clean_events').fetchone()[0]),
    'event_type_counts': {'telemetry': int(event_type_counts.get('telemetry', 0)), 'status': int(event_type_counts.get('status', 0)), 'network': int(event_type_counts.get('network', 0))},
    'period_start': con.execute('SELECT CAST(MIN(event_ts) AS VARCHAR) FROM clean_events').fetchone()[0],
    'period_end': con.execute('SELECT CAST(MAX(event_ts) AS VARCHAR) FROM clean_events').fetchone()[0],
    'device_activity_rows': int(device_activity_df.height),
    'hourly_metric_rows': int(hourly_metric_df.height),
    'activity_indicator_rows': int(activity_indicators_df.height),
}
with SUMMARY_PATH.open('w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
summary


In [ ]:
for path in [DEVICE_ACTIVITY_PATH, HOURLY_METRIC_PATH, INDICATORS_PATH, SUMMARY_PATH, QUERIES_PATH]:
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(path)
    print('[OK]', path.relative_to(ROOT))


## Висновок студента

_Ваш висновок:_
